# 1. What you'll learn

        - fit Poisson counts and Gamma positive amounts with log links
- interpret multiplicative effects on the mean
- show OLS violate support and variance assumptions

        **The one question this notebook answers:** Why should counts and positive skewed amounts use a likelihood and link that respect their shape?

# 2. Setup

Statsmodels exposes the likelihood, link, deviance, and coefficients directly, which makes it ideal for seeing what changes when the target distribution changes.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import statsmodels.api as sm
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance, mean_squared_error
from sklearn.model_selection import train_test_split

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

We create two related targets from one exposure score x. Claim count follows a Poisson mean exp(0.3+0.6x). Positive claim cost follows a Gamma distribution with mean exp(1.0+0.45x). Both have known multiplicative truth.

**Small example:** If a Poisson mean is 2 and β=log(1.5), then increasing x by one multiplies the mean by e^β=1.5: the expected count becomes 3, not 2+β.

In [ ]:
rng=np.random.default_rng(SEED);n=1200;x=rng.uniform(-2,2,n)
count_mean=np.exp(.3+.6*x);count=rng.poisson(count_mean)
cost_mean=np.exp(1+.45*x);shape=2.5;cost=rng.gamma(shape,cost_mean/shape)
frame=pd.DataFrame({"exposure_score":x,"claim_count":count,"claim_cost":cost,"true_count_mean":count_mean,"true_cost_mean":cost_mean})
print("Shape:",frame.shape);print("Zero-count share:",round((count==0).mean(),3));print("Minimum cost:",round(cost.min(),3));print(frame.head(3).round(3))

# 4. Explore

Counts pile at small integers and their spread grows with the mean. Gamma amounts stay above zero and are right-skewed. These are structural facts, not cosmetic histogram choices.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.8))
sns.histplot(frame.claim_count,discrete=True,ax=axes[0],color="#176b66");axes[0].set_title("Count target")
sns.histplot(frame.claim_cost,bins=30,ax=axes[1],color="#d38b45");axes[1].set_title("Positive skewed cost")
axes[2].scatter(frame.exposure_score,frame.claim_cost,s=9,alpha=.25,label="cost");axes[2].plot(np.sort(x),np.exp(1+.45*np.sort(x)),color="black",label="true mean");axes[2].set(title="Variance grows with mean",xlabel="x",ylabel="cost");axes[2].legend()
plt.tight_layout();plt.show()

**Takeaway:** Counts are discrete and include many zeros, which a Gaussian model does not encode.

**Takeaway:** Costs are strictly positive and strongly right-skewed.

**Takeaway:** A log-linked mean can grow multiplicatively while never becoming negative.

# 5. Prepare

We reserve 25% for test. The only design transformation is an intercept column, added separately after splitting. No response-derived transformation is fitted on held-out rows.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
train,test=train_test_split(frame,test_size=.25,random_state=SEED)
Xtr=sm.add_constant(train[["exposure_score"]]);Xte=sm.add_constant(test[["exposure_score"]])
print("Train/test:",Xtr.shape,Xte.shape);print("Count means:",round(train.claim_count.mean(),3),round(test.claim_count.mean(),3))

# 6. Train

        A GLM keeps a linear predictor but changes the target likelihood and link. The log link makes the linear score equal log mean; exponentiating produces a valid positive expectation.

        Conceptually, fitting follows these steps:

        1. Form intercept + βx as a linear predictor.
2. Use the log link so mean μ=exp(intercept+βx).
3. Compute Poisson or Gamma likelihood under that mean.
4. Iteratively reweight observations and update coefficients until deviance stops improving.

In [ ]:
poisson=sm.GLM(train.claim_count,Xtr,family=sm.families.Poisson(sm.families.links.Log())).fit()
gamma=sm.GLM(train.claim_cost,Xtr,family=sm.families.Gamma(sm.families.links.Log())).fit()
ols_count=sm.OLS(train.claim_count,Xtr).fit();ols_cost=sm.OLS(train.claim_cost,Xtr).fit()
print("Poisson coefficients:",poisson.params.round(3).to_dict(),"multipliers",np.exp(poisson.params).round(3).to_dict())
print("Gamma coefficients:",gamma.params.round(3).to_dict(),"multipliers",np.exp(gamma.params).round(3).to_dict())

# 7. Evaluate

Deviance compares predictions using the matching likelihood. We also plot fitted mean curves against known truth and inspect Pearson residuals for remaining structure.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
p_count=np.asarray(np.clip(poisson.predict(Xte),1e-6,None));p_cost=np.asarray(np.clip(gamma.predict(Xte),1e-6,None))
base_count=np.repeat(train.claim_count.mean(),len(test));base_cost=np.repeat(train.claim_cost.mean(),len(test))
print(f"Poisson deviance model={mean_poisson_deviance(test.claim_count,p_count):.3f}, baseline={mean_poisson_deviance(test.claim_count,base_count):.3f}")
print(f"Gamma deviance model={mean_gamma_deviance(test.claim_cost,p_cost):.3f}, baseline={mean_gamma_deviance(test.claim_cost,base_cost):.3f}")
order=np.argsort(test.exposure_score.to_numpy());fig,axes=plt.subplots(1,3,figsize=(14,3.8))
axes[0].scatter(test.exposure_score,test.claim_count,s=10,alpha=.3);axes[0].plot(test.exposure_score.to_numpy()[order],p_count[order],color="#176b66",label="Poisson");axes[0].plot(test.exposure_score.to_numpy()[order],test.true_count_mean.to_numpy()[order],"--",color="black",label="truth");axes[0].set_title("Count mean");axes[0].legend()
axes[1].scatter(test.exposure_score,test.claim_cost,s=10,alpha=.25);axes[1].plot(test.exposure_score.to_numpy()[order],p_cost[order],color="#d38b45",label="Gamma");axes[1].plot(test.exposure_score.to_numpy()[order],test.true_cost_mean.to_numpy()[order],"--",color="black",label="truth");axes[1].set_title("Cost mean");axes[1].legend()
pearson=(test.claim_count.to_numpy()-p_count)/np.sqrt(p_count);axes[2].scatter(p_count,pearson,s=10,alpha=.35);axes[2].axhline(0,ls="--",color="black");axes[2].set(title="Poisson Pearson residuals",xlabel="fitted mean",ylabel="scaled residual")
plt.tight_layout();plt.savefig(Path.cwd()/"key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** Both GLMs recover the multiplicative mean and beat constant baselines while respecting nonnegative support; the residual scale accounts for changing Poisson variance.

# 8. Break it

We extrapolate beyond the observed x range. An OLS line has no support constraint and can predict negative counts or costs; log-linked GLMs remain positive by construction.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
extreme=pd.DataFrame({"const":1.0,"exposure_score":np.linspace(-8,5,100)})
ols_count_ext=ols_count.predict(extreme);ols_cost_ext=ols_cost.predict(extreme);glm_count_ext=poisson.predict(extreme);glm_cost_ext=gamma.predict(extreme)
print(f"OLS minimum count prediction={ols_count_ext.min():.3f}; Poisson minimum={glm_count_ext.min():.3f}")
print(f"OLS minimum cost prediction={ols_cost_ext.min():.3f}; Gamma minimum={glm_cost_ext.min():.3f}")
print("Invalid negative OLS predictions:",int((ols_count_ext<0).sum()+(ols_cost_ext<0).sum()))

**Use this when…** the target is a count or positive right-skewed amount and multiplicative effects are a defensible description of its conditional mean.

        **Don't use this when…** counts have unexplained excess zeros, severe overdispersion, dependence, or exposure differences that the specification ignores.

        ## Try this

        1. Increase the count mean while holding β fixed; Poisson variance should rise with it.
2. Add a log exposure offset to model counts observed for different durations.
3. Generate negative-binomial counts and see Poisson residual overdispersion.